In [2]:
import ast
import json
import re
import requests

import pandas as pd
import numpy as np

from bs4 import BeautifulSoup

pd.set_option('display.max_columns', 500)

url = 'https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC' # Mapa de la Comunidad de Madrid y alrededores
df = pd.read_csv('../data/pisos_madrid.csv', sep ='|')

# Extracción de datos

In [3]:
def extraer_informacion(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    estate_component = soup.find("estate-show-v2")
    if not estate_component:
        return []
    
    estate_json_raw = estate_component.get(":estate")
    estate_json_raw = estate_json_raw.replace("&quot;", '"')
    estate_data = json.loads(estate_json_raw)

    data = {
        'dormitorios': estate_data.get('rooms'),
        'superficie_m2': estate_data.get('numeric_surface'),
        'baños': estate_data.get('bathrooms'),
        'url': estate_data.get('detail_url'),
        'features': estate_data.get('features'),
        'descripcion': estate_data.get('description'),
        'precio': estate_data.get('costs'),
        'latitud': estate_data.get('latitude'),
        'longitud': estate_data.get('longitude'),
        'media': estate_data.get('media'),
        'points_of_interest': estate_data.get('points_of_interest'),
        'energy_data': estate_data.get('energy_data'),
    }
    
    return data

In [4]:
def obtener_urls(url:str) -> list:
    '''
    Función para extraer la información relevante de la página de un anuncio.
    '''

    print(f'Buscando pisos en la página {url} ...')
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    estates_index = soup.find("estates-index")
    estates_raw = estates_index.get(":estates")
    estates_raw = estates_raw.replace("&quot;", '"')
    estates_data = json.loads(estates_raw)

    data = []

    for estate in estates_data:
        url_piso = estate.get("detail_url")

        if url_piso in df['url'].to_list():
            print('Ya lo tengo:', url_piso)
            return data
        data.append(extraer_informacion(url_piso))

    return data

In [5]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")
ultima = soup.find('a', string='>>')
max_pages = int(re.findall(r'pag-(\d+)', str(ultima))[0])

url_splited = url.split('pag-1')
data = []
for i in range(1, max_pages+1):
    subdata = obtener_urls(f'pag-{i}'.join(url_splited))
    data.extend(subdata)

    if len(subdata) < 15:
        break

df = pd.concat([pd.DataFrame(data), df])
df.to_csv('../data/pisos_madrid.csv', sep='|', index=False)

Buscando pisos en la página https://www.tecnocasa.es/venta/piso/mapa.html/pag-1?view=41.18051487737182,-2.109104143945018,39.64144152264612,-5.319858538475728&zoom=9&sort=Aggiornamento|DESC ...
Ya lo tengo: https://www.tecnocasa.es/venta/piso/madrid/las-rozas-de-madrid/648733.html


# Pretratamiento de datos

In [6]:
# A partir de este punto sería interesante integrar un pipeline, para que si entra un nuevo piso todo se ejecute en el orden adecuado.

In [7]:
df[['features', 'precio', 'media', 'points_of_interest', 'energy_data']] = df[['features', 'precio', 'media', 'points_of_interest', 'energy_data']].fillna('{}')

df['planta'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('floor'))
df['aire_acondicionado'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('air_conditioning'))
df['ascensor'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('elevator'))
df['calefaccion'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('heating'))
df['categoria'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('category'))
df['año_construccion'] = df['features'].apply(ast.literal_eval).apply(lambda x: x.get('build_year'))

df['precio'] = df['precio'].replace(' €', '', regex=True)
df['precio_euros'] = df['precio'].apply(ast.literal_eval).apply(lambda x: x.get('price'))
df['hipoteca'] = df['precio'].apply(ast.literal_eval).apply(lambda x: x.get('mortgage_payment_value'))

df['planos'] = df['media'].apply(ast.literal_eval).apply(lambda x: x.get('floor_plans') != None)
df['realistico'] = df['media'].apply(ast.literal_eval).apply(lambda x: x.get('has_realistico'))
df['fotografias'] = df['media'].apply(ast.literal_eval).apply(lambda x: len(x.get('images')))

df['transporte_publico'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('public_transport'))
df['escuelas'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('school'))
df['farmacias'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('pharmacy'))
df['hospitales'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('hospital'))
df['supermercados'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('market'))
df['tiendas'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('shop'))
df['bares'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('bar'))
df['restaurantes'] = df['points_of_interest'].apply(ast.literal_eval).apply(lambda x: x.get('restaurant'))

df['clase_energetica'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('class_emissions'))
df['eficiencia_energetica'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('efficiency'))
df['emisiones_energeticas'] = df['energy_data'].apply(ast.literal_eval).apply(lambda x: x.get('emissions'))

In [8]:
df.head()

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,transporte_publico,escuelas,farmacias,hospitales,supermercados,tiendas,bares,restaurantes,clase_energetica,eficiencia_energetica,emisiones_energeticas
0,1 dorm.,50.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/las...,"{'id': 648733, 'floor': '', 'floors': None, 'b...",<p>Inmobiliaria en Las Rozas De Madrid - Tecno...,"{'price': '327.500', 'box_price': None, 'mortg...",40.492402,-3.878863,"{'floor_plans': None, 'has_realistico': False,...",{},"{'class': '', 'class_emissions': None, 'certif...",,None,,Independiente (Gas),Popular,1991,327.500,1.055,False,False,36,None,None,None,None,None,None,None,None,None,None,None
1,3 dorm.,91.0,2 baños,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 649824, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '565.000', 'box_price': None, 'mortg...",40.480702,-3.713013,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,None,Sí,centralizada (Gas),Popular,1974,565.000,1.820,False,False,40,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Peque's School', 'class': 'school',...",[{'name': 'Farmacia - Calle Fermín Caballero 7...,"[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Supersol', 'class': 'grocery', 'sub...","[{'name': 'La Tahona', 'class': 'shop', 'subcl...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'La Mesa Puesta', 'class': 'restaura...",d,172 kW h/m&sup2; año,36
2,7 dorm.,190.0,3 baños,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 640542, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '990.000', 'box_price': None, 'mortg...",40.480602,-3.717203,"{'floor_plans': [{'id': 9600945, 'width': 683,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,None,,Independiente (Gas),Señorial,1968,990.000,3.189,True,False,30,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Colegio Público Bravo Murillo', 'cl...","[{'name': 'Farmacia - Calle Isla de Arosa 43',...","[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Unide Market', 'class': 'grocery', ...","[{'name': 'Tiendas', 'class': 'shop', 'subclas...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'Bar Restaurante La Ponderosa', 'cla...",d,"124,2 kWh/m&sup2; año","25,8"
3,3 dorm.,68.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 598798, 'floor': '', 'floors': None, 'b...","<p>Reforma a estrenar, estancias exteriores e ...","{'price': '290.000', 'box_price': None, 'mortg...",40.432702,-3.640053,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Ascao', 'class...","{'class': '', 'class_emissions': None, 'certif...",,None,,"Independiente (Gas, Radiadores)",Media,1960,290.000,934,False,False,24,"[{'name': 'Ascao', 'class': 'railway', 'subcla...","[{'name': 'Colegio Joyfe', 'class': 'school', ...","[{'name': 'Farmacia - Calle Vital Aza 59', 'cl...","[{'name': 'Clínica Fuensanta', 'class': 'hospi...","[{'name': 'Ahorramás', 'class': 'grocery', 'su...","[{'name': 'Levain', 'class': 'shop', 'subclass...","[{'name': 'Los Tres Diamantes', 'class': 'bar'...","[{'name': 'Piscis', 'class': 'restaurant', 'su...",None,None,None
4,3 dorm.,90.0,NaN,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 648942, 'floor': '', 'floors': None, 'b...",<p>Agencia inmobiliaria de MADRID - zona VALLE...,"{'price': '249.000', 'box_price': None, 'mortg...",40.391302,-3.660652,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Portazgo', 'cl...","{'class': '', 'class_emiss

In [9]:
df['tp_cnt'] = df['transporte_publico'].apply(lambda lst: len(lst) if lst else 0)
tp_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['tp_min_dist_m'] = df['transporte_publico'].apply(lambda lst: np.nan if not lst else min(tp_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['esc_cnt'] = df['escuelas'].apply(lambda lst: len(lst) if lst else 0)
esc_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['esc_min_dist_m'] = df['escuelas'].apply(lambda lst: np.nan if not lst else min(esc_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['fca_cnt'] = df['farmacias'].apply(lambda lst: len(lst) if lst else 0)
fca_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['fca_min_dist_m'] = df['farmacias'].apply(lambda lst: np.nan if not lst else min(fca_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['hosp_cnt'] = df['hospitales'].apply(lambda lst: len(lst) if lst else 0)
hosp_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['hosp_min_dist_m'] = df['hospitales'].apply(lambda lst: np.nan if not lst else min(hosp_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['super_cnt'] = df['supermercados'].apply(lambda lst: len(lst) if lst else 0)
super_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['super_min_dist_m'] = df['supermercados'].apply(lambda lst: np.nan if not lst else min(super_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['tda_cnt'] = df['tiendas'].apply(lambda lst: len(lst) if lst else 0)
tda_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['tda_min_dist_m'] = df['tiendas'].apply(lambda lst: np.nan if not lst else min(tda_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['bar_cnt'] = df['bares'].apply(lambda lst: len(lst) if lst else 0)
bar_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['bar_min_dist_m'] = df['bares'].apply(lambda lst: np.nan if not lst else min(bar_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))

df['resto_cnt'] = df['restaurantes'].apply(lambda lst: len(lst) if lst else 0)
resto_m = lambda s: (lambda t: float(t.replace('km',''))*1000 if 'km' in t else float(t.replace('m','')))(str(s).lower().replace(',','.').replace(' ',''))
df['resto_min_dist_m'] = df['restaurantes'].apply(lambda lst: np.nan if not lst else min(resto_m(d.get('distance')) for d in lst if isinstance(d, dict) and d.get('distance')))


In [11]:
df.head(1)

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,transporte_publico,escuelas,farmacias,hospitales,supermercados,tiendas,bares,restaurantes,clase_energetica,eficiencia_energetica,emisiones_energeticas,tp_cnt,tp_min_dist_m,esc_cnt,esc_min_dist_m,fca_cnt,fca_min_dist_m,hosp_cnt,hosp_min_dist_m,super_cnt,super_min_dist_m,tda_cnt,tda_min_dist_m,bar_cnt,bar_min_dist_m,resto_cnt,resto_min_dist_m
0,1 dorm.,50.0,1 baño,https://www.tecnocasa.es/venta/piso/madrid/las...,"{'id': 648733, 'floor': '', 'floors': None, 'b...",<p>Inmobiliaria en Las Rozas De Madrid - Tecno...,"{'price': '327.500', 'box_price': None, 'mortg...",40.492402,-3.878863,"{'floor_plans': None, 'has_realistico': False,...",{},"{'class': '', 'class_emissions': None, 'certif...",,None,,Independiente (Gas),Popular,1991,327.500,1.055,False,False,36,None,None,None,None,None,None,None,None,None,None,None,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN


In [10]:
# Aquí extraemos la información de las columnas de puntos de interes: Total de elementos por columna y distancia del más cercano.

In [11]:
# La columna de descripción podría tener información valiosa si le aplicamos un LLM (HuggingFace)

In [14]:
df = df.drop(columns=['url', 'features', 'descripcion', 'precio', 'media', 'points_of_interest', 'energy_data', 'transporte_publico', 'escuelas', 'farmacias', 'hospitales', 'supermercados', 'tiendas', 'bares', 'restaurantes'])

In [15]:
df.head()

,dormitorios,superficie_m2,baños,latitud,longitud,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,clase_energetica,eficiencia_energetica,emisiones_energeticas,tp_cnt,tp_min_dist_m,esc_cnt,esc_min_dist_m,fca_cnt,fca_min_dist_m,hosp_cnt,hosp_min_dist_m,super_cnt,super_min_dist_m,tda_cnt,tda_min_dist_m,bar_cnt,bar_min_dist_m,resto_cnt,resto_min_dist_m,calefaccion_gas,calefaccion_electrica
0,1,50.0,1,40.492402,-3.878863,,None,,Independiente,Popular,1991,327.500,1.055,False,False,36,None,None,None,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,True,False
1,3,91.0,2,40.480702,-3.713013,Baja,None,Sí,centralizada,Popular,1974,565.000,1.820,False,False,40,d,172,36,5,110.0,5,120.0,5,130.0,5,950.0,5,140.0,5,70.0,5,70.0,5,50.0,True,False
2,7,190.0,3,40.480602,-3.717203,Baja,None,,Independiente,Señorial,1968,990.000,3.189,True,False,30,d,124.2,25.8,5,20.0,5,110.0,5,30.0,5,940.0,5,40.0,5,130.0,5,380.0,5,80.0,True,False
3,3,68.0,1,40.432702,-3.640053,,None,,Independiente,Media,1960,290.000,934,False,False,24,None,None,None,4,220.0,5,60.0,5,90.0,5,720.0,5,140.0,5,290.0,5,310.0,5,330.0,True,False
4,3,90.0,NaN,40.391302,-3.660652,,None,,None,Media,1956,249.000,802,False,False,17,None,None,None,5,60.0,5,150.0,5,90.0,2,2800.0,5,340.0,5,300.0,5,50.0,5,270.0,None,None


# Limpieza de datos

In [14]:
# from sklearn.impute import SimpleImputer

# imputer = SimpleImputer(strategy='median')

# imputer.fit_transform(pd.DataFrame(df['dormitorios'].replace(' dorm.', '', regex=True)))

In [12]:
df['dormitorios'] = df['dormitorios'].replace(' dorm.', '', regex=True)
df['baños'] = df['baños'].replace(' baño[s]*', '', regex=True)
df['planta'] = df['planta'].replace(r'\s*\(.*\)', '', regex=True)
df['aire_acondicionado'] = df['aire_acondicionado'].replace(r'\s+.+', '', regex=True)
df['calefaccion_gas'] = df['calefaccion'].str.contains('gas', case=False)
df['calefaccion_electrica'] = df['calefaccion'].str.contains('eléctrica', case=False)
df['calefaccion'] = df['calefaccion'].replace(r'\s+.+', '', regex=True)
df['eficiencia_energetica'] = df['eficiencia_energetica'].replace(r'\s.*', '', regex=True).replace(',', '.', regex=True)
df['emisiones_energeticas'] = df['emisiones_energeticas'].replace(',', '.', regex=True)

In [13]:
df.head()

,dormitorios,superficie_m2,baños,url,features,descripcion,precio,latitud,longitud,media,points_of_interest,energy_data,planta,aire_acondicionado,ascensor,calefaccion,categoria,año_construccion,precio_euros,hipoteca,planos,realistico,fotografias,transporte_publico,escuelas,farmacias,hospitales,supermercados,tiendas,bares,restaurantes,clase_energetica,eficiencia_energetica,emisiones_energeticas,tp_cnt,tp_min_dist_m,esc_cnt,esc_min_dist_m,fca_cnt,fca_min_dist_m,hosp_cnt,hosp_min_dist_m,super_cnt,super_min_dist_m,tda_cnt,tda_min_dist_m,bar_cnt,bar_min_dist_m,resto_cnt,resto_min_dist_m,calefaccion_gas,calefaccion_electrica
0,1,50.0,1,https://www.tecnocasa.es/venta/piso/madrid/las...,"{'id': 648733, 'floor': '', 'floors': None, 'b...",<p>Inmobiliaria en Las Rozas De Madrid - Tecno...,"{'price': '327.500', 'box_price': None, 'mortg...",40.492402,-3.878863,"{'floor_plans': None, 'has_realistico': False,...",{},"{'class': '', 'class_emissions': None, 'certif...",,None,,Independiente,Popular,1991,327.500,1.055,False,False,36,None,None,None,None,None,None,None,None,None,None,None,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,True,False
1,3,91.0,2,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 649824, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '565.000', 'box_price': None, 'mortg...",40.480702,-3.713013,"{'floor_plans': None, 'has_realistico': False,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,None,Sí,centralizada,Popular,1974,565.000,1.820,False,False,40,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Peque's School', 'class': 'school',...",[{'name': 'Farmacia - Calle Fermín Caballero 7...,"[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Supersol', 'class': 'grocery', 'sub...","[{'name': 'La Tahona', 'class': 'shop', 'subcl...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'La Mesa Puesta', 'class': 'restaura...",d,172,36,5,110.0,5,120.0,5,130.0,5,950.0,5,140.0,5,70.0,5,70.0,5,50.0,True,False
2,7,190.0,3,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 640542, 'floor': 'Baja', 'floors': None...",<p>Agencia inmobiliaria de MADRID - zona Aveni...,"{'price': '990.000', 'box_price': None, 'mortg...",40.480602,-3.717203,"{'floor_plans': [{'id': 9600945, 'width': 683,...",{'public_transport': [{'name': 'Avenida de la ...,"{'class': 'd', 'class_emissions': 'd', 'certif...",Baja,None,,Independiente,Señorial,1968,990.000,3.189,True,False,30,"[{'name': 'Avenida de la Ilustración', 'class'...","[{'name': 'Colegio Público Bravo Murillo', 'cl...","[{'name': 'Farmacia - Calle Isla de Arosa 43',...","[{'name': 'Clínica Ruber Internacional', 'clas...","[{'name': 'Unide Market', 'class': 'grocery', ...","[{'name': 'Tiendas', 'class': 'shop', 'subclas...","[{'name': 'Hemeya', 'class': 'bar', 'subclass'...","[{'name': 'Bar Restaurante La Ponderosa', 'cla...",d,124.2,25.8,5,20.0,5,110.0,5,30.0,5,940.0,5,40.0,5,130.0,5,380.0,5,80.0,True,False
3,3,68.0,1,https://www.tecnocasa.es/venta/piso/madrid/mad...,"{'id': 598798, 'floor': '', 'floors': None, 'b...","<p>Reforma a estrenar, estancias exteriores e ...","{'price': '290.000', 'box_price': None, 'mortg...",40.432702,-3.640053,"{'floor_plans': None, 'has_realistico': False,...","{'public_transport': [{'name': 'Ascao', 'class...","{'class': '', 'class_emissions': None, 'certif...",,None,,Independiente,Media,1960,290.000,934,False,False,24,"[{'name': 'Ascao', 'class': 'railway', 'subcla...","[{'name': 'Colegio Joyfe', 'class': 'school', ...","[{'name': 'Farmacia - Calle Vital Aza 59', 'cl...","[{'name': 'Clínica Fuensanta', 'class': 'hospi...","[{'name': 'Ahorramás', 'class': 'grocery', 'su...","[{'name': 'Levain', 'class': 'shop', 'subclass...","[{'name': 'Los Tres Diamantes', 'class': 'bar'...","[{'name': 'Piscis', 'class': 'restaurant', 'su...",None,None,None,4,220.0,5,60.0,5,90.0,5,720.0,5,140.0,5,290.0,5,310.0,5,330.0,T

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dormitorios            1097 non-null   object 
 1   superficie_m2          1208 non-null   float64
 2   baños                  1111 non-null   object 
 3   latitud                1208 non-null   float64
 4   longitud               1208 non-null   float64
 5   planta                 1208 non-null   object 
 6   aire_acondicionado     450 non-null    object 
 7   ascensor               1208 non-null   object 
 8   calefaccion            670 non-null    object 
 9   categoria              1208 non-null   object 
 10  año_construccion       1208 non-null   int64  
 11  precio_euros           1208 non-null   object 
 12  hipoteca               1208 non-null   object 
 13  planos                 1208 non-null   bool   
 14  realistico             1208 non-null   bool   
 15  foto

In [18]:
# Una vez tengamos nuestro modelo, hacer modelos con menos features, valorando la mejora de rendimiento/peso del modelo, frente a la perdida de precisión.